# YouTube Data Pipeline: Pemberitaan Program Makan Bergizi Gratis (MBG)

Notebook ini mengambil **metadata video** tentang program Makan Bergizi Gratis (MBG) dari 28 channel YouTube
(27 media nasional + channel resmi Badan Gizi Nasional) melalui **YouTube Data API v3**, lalu membersihkannya
dan mengekstraksi lokasi yang disebut di judul/deskripsi. Hasilnya menjadi input `scripts/build_star_schema.py`
dan dashboard Power BI di repo ini.

## Ringkasan Pipeline

```
 Daftar 28 channel (27 media nasional + BGN)
            │
            ▼
 [1.7] channels.list ──► info channel, uploads playlist, jumlah video
            │
            ├── channel ≤ 20.000 video ──► [1.8] playlistItems.list (uploads playlist)
            └── channel > 20.000 video ──► [1.9] search.list (keyword MBG per kuartal)
            │
            ▼
 [1.11] Gabung kandidat + hapus duplikat video_id
            │
            ▼
 [1.12] videos.list (batch 50) ──► [1.13] validasi channel, periode, keyword MBG
            │
            ▼
 [2] Pembersihan + [2.4] ekstraksi lokasi (kamus wilayah)
            │
            ▼
 data/source/videos_clean.csv ──► scripts/build_star_schema.py ──► Power BI
```

Semua langkah memakai **YouTube Data API v3** melalui library `requests`. Kuota harian API terbatas (10.000 unit
untuk endpoint umum dan 100 panggilan `search.list`), sehingga pengumpulan dapat memerlukan beberapa hari.
Setiap progres disimpan di folder `data/raw/youtube/state/`. Jika kuota habis, jalankan ulang notebook keesokan
harinya, maka proses **melanjutkan dari titik terakhir (resume)**.

**Cakupan data:** notebook ini hanya mengambil metadata video publik dari channel media dan lembaga resmi.
Komentar penonton **tidak** diambil (lihat bagian 4).

# 1. Pengumpulan Data

## 1.1. Import Library

Mengimpor library yang dibutuhkan untuk keseluruhan proses pengambilan dan pengolahan data, yaitu:

* `requests` untuk melakukan HTTP request ke YouTube Data API.
* `pandas` dan `numpy` untuk mengolah dan menyimpan data dalam bentuk tabel (DataFrame/CSV).
* `matplotlib` dan `wordcloud` untuk visualisasi pada bagian EDA.
* `json` untuk menyimpan data mentah dan progres pengumpulan.
* `time` untuk memberi jeda sebelum mencoba ulang request yang gagal.
* `html`, `unicodedata`, dan `re` untuk membersihkan teks.
* `python-dotenv` (diimpor saat membaca API key) untuk membaca API key dari file `.env`.

In [ ]:
import os
import re
import json
import time
import html
import unicodedata
from collections import Counter
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_colwidth", 80)

## 1.2. Konfigurasi Penelitian

Mendefinisikan seluruh parameter penelitian dalam satu sel agar mudah diperiksa dan dilaporkan:

* `PERIOD_START` dan `PERIOD_END`: periode **tanggal terbit video** dalam WIB, mulai 6 Januari 2025 (awal pelaksanaan MBG) sampai 30 September 2026.
* `MBG_PATTERN`: regex untuk memvalidasi judul/deskripsi video. Video dianggap relevan jika memuat kata `MBG` atau frasa `makan bergizi gratis` / `makanan bergizi gratis`. Pola dibuat luas agar dataset induk mencakup seluruh pembahasan MBG; pemilahan topik keamanan pangan dilakukan pada tahap analisis.
* `SEARCH_QUERY`: query untuk `search.list`. Operator `|` berarti ATAU.
* `MAX_UNITS_PER_SESSION` dan `MAX_SEARCH_CALLS_PER_SESSION`: pengaman agar satu sesi tidak menghabiskan seluruh kuota harian.
* `COLLECT_DATA`: `True` untuk mengambil/melanjutkan data dari API; `False` untuk hanya menjalankan preprocessing dan EDA pada data yang sudah tersimpan (tidak memerlukan API key).

In [ ]:
PERIOD_START = "2025-01-06"
PERIOD_END = "2026-09-30"  # inklusif

MBG_PATTERN = r"\bmbg\b|\bmakan(an)?\s+bergizi\s+gratis\b"
SEARCH_QUERY = 'MBG|"makan bergizi gratis"'

MAX_UNITS_PER_SESSION = 9000        # kuota default harian: 10.000 unit
MAX_SEARCH_CALLS_PER_SESSION = 100  # kuota default harian: 100 panggilan search.list

COLLECT_DATA = True

In [ ]:
# API YouTube memakai waktu UTC, jadi batas periode WIB (UTC+7) dikonversi ke UTC.
START_UTC = pd.Timestamp(f"{PERIOD_START}T00:00:00+07:00").tz_convert("UTC")
END_UTC = pd.Timestamp(f"{PERIOD_END}T00:00:00+07:00").tz_convert("UTC") + pd.Timedelta(days=1)
MBG_REGEX = re.compile(MBG_PATTERN, re.IGNORECASE)

print("Periode video (UTC):", START_UTC, "sampai sebelum", END_UTC)

## 1.3. Daftar Channel Sumber

Mendefinisikan daftar channel YouTube yang menjadi sumber data, yaitu **27 channel media nasional** dan **1 channel resmi Badan Gizi Nasional (BGN)**. Alasan pemilihan:

* Media nasional memberitakan insiden MBG dari berbagai daerah, sehingga sebaran wilayah tetap terwakili tanpa perlu mengumpulkan puluhan channel daerah.
* Channel BGN mewakili komunikasi resmi pemerintah dan **dibedakan** dari media melalui kolom `category`.
* Setiap `channel_id` sudah diverifikasi dari situs resmi penerbit atau dikonfirmasi peneliti (bukti tersimpan di `youtube_channel_registry.json`), sehingga channel tiruan atau *fan page* tidak ikut terambil.

In [ ]:
CHANNELS = [
    # (record_id, channel_name, channel_id, category)
    ("N001", "KOMPASTV", "UC5BMIWZe9isJXLZZWPWvBlg", "national_media"),
    ("N002", "CNN Indonesia", "UCKII0Ml9S5wneKbHswmUrIQ", "national_media"),
    ("N003", "METRO TV", "UCzl0OrB3-ehunyotIQvK77A", "national_media"),
    ("N004", "tvOneNews", "UCER4rvDnRBPr_ncYW4UCZjg", "national_media"),
    ("N005", "Official iNews", "UCoSkllfpgmFHtbVK835QaQg", "national_media"),
    ("N006", "BeritaSatu", "UCqLsfkQSM0yfyGvONAGWd3Q", "national_media"),
    ("N007", "TVRI Nasional", "UCyIMsCzDRm8_y_6wM26_PWQ", "national_media"),
    ("N008", "RRI NET OFFICIAL", "UCyUr8PM0PFwYQAy5yUPH36Q", "national_media"),
    ("N009", "GARUDA TV", "UCmXp0n4Oq8DGzjRZj3EvG4Q", "national_media"),
    ("N010", "CNBC Indonesia", "UCGN9JsnkvK05v2lnTI_-uGA", "national_media"),
    ("N011", "Kompas.com", "UCPAxpUn1mrn14xU0JpsLhDg", "national_media"),
    ("N012", "detikcom", "UCuMAjEaSMj7q7YLf0xW1MjQ", "national_media"),
    ("N013", "Tribunnews", "UCmxAIW7RDDC88EPk4ry16Kg", "national_media"),
    ("N014", "Liputan6", "UC1QpnGo11epfexw1bIiTHTg", "national_media"),
    ("N015", "kumparan", "UCm2Fz7aL464TC_0uG_yagng", "national_media"),
    ("N016", "Antara TV Indonesia", "UCAXcJqVpDHSfr_MdEuclSCg", "national_media"),
    ("N017", "Tempodotco", "UC3QRoNY-nYDTNSv-1dR0P-g", "national_media"),
    ("N018", "Narasi Newsroom", "UCnOf30K0d0e8M7mc4FUzR0A", "national_media"),
    ("N019", "IDN TIMES", "UCMcqmnF-uCBFzQ3NvmaELCg", "national_media"),
    ("N020", "Suaradotcom", "UCeq1WeHIbMyeKp_sUNolgzA", "national_media"),
    ("N021", "medcom id", "UCjEz1Z4kKpb5T6HzFw_qlzA", "national_media"),
    ("N022", "JPNN.COM", "UCHYuEi41c1RcoTf8NgJLdkg", "national_media"),
    ("N023", "MerdekaDotCom", "UCdOovgmIPfj9a-o3VP5i2_g", "national_media"),
    ("N024", "SINDOnews", "UCGfXjFgIUUTUUlzdG6BenXA", "national_media"),
    ("N025", "REPUBLIKA Official", "UCWBqcKLh_Ta6ua00OX57ejA", "national_media"),
    ("N029", "VIVA.CO.ID", "UCB1fvlA5xmLJ0tZblpoTuaQ", "national_media"),
    ("N030", "Okezone", "UCHc8PjWfjZUFVmPQeLcFT2g", "national_media"),
    ("G001", "Badan Gizi Nasional Republik Indonesia", "UC3USej-LVYCple4BOOLGYTg", "government"),
]
df_channels = pd.DataFrame(CHANNELS, columns=["record_id", "channel_name", "channel_id", "category"])

# Pastikan tidak ada channel_id ganda.
assert df_channels["channel_id"].is_unique, "Ada channel_id ganda pada daftar channel."
print(f"{len(df_channels)} channel:", df_channels["category"].value_counts().to_dict())

## 1.4. Folder Penyimpanan dan Fungsi Helper File

Menyiapkan folder penyimpanan. Notebook berada di folder `notebooks/`; semua data disimpan di folder `data/`
pada folder proyek:

| Folder | Isi | Ikut Git? |
|---|---|---|
| `data/raw/youtube/api/` | Respons asli API dalam format JSON Lines (satu baris per request), tidak pernah diubah | Tidak |
| `data/raw/youtube/state/` | Progres pencarian per channel, dipakai untuk melanjutkan (*resume*) | Tidak |
| `data/raw/youtube/logs/` | Log setiap request API (waktu, endpoint, status) untuk menghitung pemakaian kuota | Tidak |
| `data/raw/youtube/figures/` | Grafik EDA | Tidak |
| `data/source/` | Dataset video bersih dan tabel ringkasan, input `build_star_schema.py` | Tidak |

Fungsi helper `save_json()` dan `save_csv()` menulis ke file sementara terlebih dahulu lalu menggantinya
(`replace`), sehingga file tidak rusak setengah jadi jika notebook terhenti di tengah penulisan.

In [ ]:
# Folder proyek = induk dari folder notebooks/.
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
YOUTUBE_RAW_DIR = BASE_DIR / "data" / "raw" / "youtube"
RAW_DIR = YOUTUBE_RAW_DIR / "api"
STATE_DIR = YOUTUBE_RAW_DIR / "state"
LOG_DIR = YOUTUBE_RAW_DIR / "logs"
FIGURE_DIR = YOUTUBE_RAW_DIR / "figures"
OUTPUT_DIR = BASE_DIR / "data" / "source"

for folder in [RAW_DIR, STATE_DIR, LOG_DIR, FIGURE_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Folder proyek:", BASE_DIR)

In [ ]:
def now_utc():
    # Waktu UTC format ISO, untuk mencatat kapan data diambil.
    return datetime.now(timezone.utc).isoformat()


def save_json(path, data):
    tmp_path = Path(str(path) + ".tmp")
    tmp_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp_path.replace(path)


def load_json(path, default):
    return json.loads(Path(path).read_text(encoding="utf-8")) if Path(path).exists() else default


def save_csv(df, path):
    tmp_path = Path(str(path) + ".tmp")
    df.to_csv(tmp_path, index=False, encoding="utf-8")
    tmp_path.replace(path)


def load_csv(path):
    # Semua kolom dibaca sebagai teks; konversi tipe data dilakukan pada tahap preprocessing.
    if not Path(path).exists():
        return pd.DataFrame()
    return pd.read_csv(path, dtype=str, keep_default_na=False)


def append_jsonl(path, data):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(data, ensure_ascii=False) + "\n")

## 1.5. API Key YouTube

Mengambil API key YouTube **tanpa menuliskannya di dalam kode**. Fungsi `get_api_key()` mencari key secara berurutan dari:

1. *environment variable* `YT_API_KEY` atau `YOUTUBE_API_KEY`;
2. file `.env` di folder proyek (isi: `YT_API_KEY=...`, contoh di `.env.example`; file ini tidak ikut diunggah ke GitHub);
3. Colab Secret bernama `YT_API_KEY`;
4. input tersembunyi (`getpass`).

Untuk memastikan key terdeteksi, yang dicetak hanya panjang key, bukan isinya. Jika `COLLECT_DATA = False`, API key tidak diminta.

In [ ]:
def get_api_key():
    key_names = ["YT_API_KEY", "YOUTUBE_API_KEY"]
    for name in key_names:
        value = os.environ.get(name)
        if value:
            return value.strip()

    env_file = BASE_DIR / ".env"
    if env_file.exists():
        from dotenv import dotenv_values
        env_values = dotenv_values(env_file)
        for name in key_names:
            value = env_values.get(name)
            if value:
                return value.strip()

    try:
        from google.colab import userdata  # Google Colab.
        value = userdata.get("YT_API_KEY")
        if value:
            return value.strip()
    except Exception:
        pass

    return getpass("Masukkan YouTube API key (tidak disimpan): ").strip()


API_KEY = get_api_key() if COLLECT_DATA else None
if COLLECT_DATA:
    print("API key terdeteksi (disembunyikan). Panjang:", len(API_KEY))
else:
    print("COLLECT_DATA = False: pengambilan data dilewati, langsung memakai data tersimpan.")

## 1.6. Fungsi Helper untuk Memanggil YouTube Data API

Mendefinisikan fungsi `youtube_get()` sebagai *wrapper* untuk semua pemanggilan YouTube Data API v3, sehingga perilaku setiap request seragam. Fungsi ini:

* **Menghitung pemakaian kuota.** Setiap request (termasuk percobaan ulang) dihitung. Endpoint umum memakai 1 unit per request, sedangkan `search.list` dihitung terpisah karena memiliki kuota sendiri. Jika batas sesi tercapai, fungsi melempar `QuotaExceeded`, dan progres yang sudah tersimpan tetap aman.
* **Menangani error.** Error jaringan, HTTP `429`, dan `5xx` dicoba ulang hingga 3 kali dengan jeda bertambah (2 lalu 4 detik). Error `quotaExceeded` dari API menghentikan sesi. Error lain (misalnya `videoNotFound`) diteruskan sebagai `ApiError` untuk ditangani oleh fungsi pemanggil.
* **Menyimpan data mentah.** Setiap respons sukses disimpan apa adanya ke `raw/<endpoint>.jsonl` beserta parameter dan waktu pengambilan. API key tidak ikut disimpan.
* **Mencatat log** setiap request ke `logs/api_calls.csv`.

In [ ]:
class QuotaExceeded(Exception):
    # Dilempar ketika kuota harian API atau batas sesi sudah habis.
    def __init__(self, quota_type):
        self.quota_type = quota_type  # "unit" (endpoint umum) atau "search"
        super().__init__(f"Kuota {quota_type} habis")


class ApiError(Exception):
    # Error dari API yang bukan masalah kuota, misalnya video tidak ditemukan.
    def __init__(self, reason, status):
        self.reason = reason
        self.status = status
        super().__init__(f"{reason} (HTTP {status})")


usage = {"unit": 0, "search": 0}  # Penghitung request pada sesi ini.
session = requests.Session()
LOG_PATH = LOG_DIR / "api_calls.csv"


def write_log(endpoint, status, reason):
    is_new_file = not LOG_PATH.exists()
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        if is_new_file:
            f.write("time_utc,endpoint,status,reason\n")
        f.write(f"{now_utc()},{endpoint},{status},{reason}\n")


def get_error_reason(response):
    # Ambil kode alasan dari body error API, misalnya "quotaExceeded" atau "videoNotFound".
    try:
        return response.json()["error"]["errors"][0]["reason"]
    except Exception:
        return f"http{response.status_code}"


def youtube_get(endpoint, params):
    quota_type = "search" if endpoint == "search" else "unit"
    limit = MAX_SEARCH_CALLS_PER_SESSION if quota_type == "search" else MAX_UNITS_PER_SESSION
    params = {k: v for k, v in params.items() if v is not None}  # Buang parameter kosong.
    url = f"https://www.googleapis.com/youtube/v3/{endpoint}"

    for attempt in range(1, 4):
        if usage[quota_type] >= limit:
            raise QuotaExceeded(quota_type)
        usage[quota_type] += 1

        try:
            response = session.get(url, params={**params, "key": API_KEY}, timeout=30)
            status = response.status_code
        except requests.RequestException:
            status, reason = None, "network_error"  # Detail exception tidak dicetak agar key tidak bocor.

        if status == 200:
            data = response.json()
            write_log(endpoint, status, "")
            append_jsonl(RAW_DIR / f"{endpoint}.jsonl",
                         {"retrieved_at_utc": now_utc(), "params": params, "response": data})
            return data

        if status is not None:
            reason = get_error_reason(response)
        write_log(endpoint, status, reason)

        if reason in ("quotaExceeded", "dailyLimitExceeded"):
            raise QuotaExceeded(quota_type)
        retryable = status is None or status == 429 or status >= 500
        if not retryable or attempt == 3:
            raise ApiError(reason, status)
        time.sleep(2 ** attempt)  # Jeda 2 lalu 4 detik sebelum mencoba lagi.

## 1.7. Informasi Channel dan Pemilihan Metode Pencarian Video

Mengambil informasi setiap channel melalui endpoint `channels.list` (part `snippet,statistics,contentDetails`): nama channel, jumlah video, dan ID **uploads playlist**. Ke-28 ID channel dikirim dalam satu request, karena satu request menerima hingga 50 ID.

**Mengapa ada dua metode pencarian video?**

Setiap channel YouTube memiliki satu playlist otomatis bernama *Uploads* yang berisi **semua video channel tersebut, diurutkan dari yang terbaru**. Membaca playlist ini adalah cara paling lengkap dan hemat kuota, karena semua video diperiksa satu per satu, tidak bergantung pada hasil pencarian YouTube.

Masalahnya, **API hanya mengizinkan pembacaan sekitar 20.000 video terbaru** dari playlist ini. Pada run uji coba (v1.2), terlihat:

| Channel | Jumlah video di channel | Yang bisa dibaca lewat playlist | Upload per hari (perkiraan) | Video tertua yang terjangkau |
|---|---|---|---|---|
| KOMPASTV | 417.561 | 19.979 | ±128 | sekitar Mei 2026 |
| METRO TV | 438.775 | 20.000 | ±328 | sekitar Agustus 2026 |
| Tribunnews | 464.043 | 20.000 | ±264 | sekitar Agustus 2026 |
| Narasi Newsroom | 5.015 | 5.003 | ±1 | seluruh video |

Artinya, untuk channel berita besar, video MBG awal 2025 **tidak akan pernah tercapai** lewat playlist. Karena itu, metode dipilih otomatis berdasarkan jumlah video channel:

* **≤ 20.000 video → `playlist`**: semua upload dalam periode bisa ditelusuri lewat uploads playlist (bagian 1.8).
* **> 20.000 video → `search`**: memakai `search.list` dengan filter channel, keyword MBG, dan rentang waktu (bagian 1.9).

Hasil disimpan ke `state/channel_info.csv` sehingga tidak perlu diminta ulang pada sesi berikutnya.

In [ ]:
PLAYLIST_LIMIT = 20_000  # Batas jumlah video yang bisa dibaca dari uploads playlist.


def get_channel_info(channel_ids):
    data = youtube_get("channels", {
        "part": "snippet,statistics,contentDetails",
        "id": ",".join(channel_ids),
    })
    rows = []
    for item in data.get("items", []):
        rows.append({
            "channel_id": item["id"],
            "channel_title_api": item["snippet"]["title"],
            "channel_video_count": int(item["statistics"].get("videoCount", 0)),
            "subscriber_count": item["statistics"].get("subscriberCount", ""),
            "uploads_playlist_id": item["contentDetails"]["relatedPlaylists"]["uploads"],
        })
    return pd.DataFrame(rows)

In [ ]:
channel_info_path = STATE_DIR / "channel_info.csv"
if channel_info_path.exists():
    channel_info = load_csv(channel_info_path)
    channel_info["channel_video_count"] = channel_info["channel_video_count"].astype(int)
elif COLLECT_DATA:
    channel_info = get_channel_info(df_channels["channel_id"].tolist())
    save_csv(channel_info, channel_info_path)
else:
    raise FileNotFoundError("Belum ada data. Jalankan dengan COLLECT_DATA = True terlebih dahulu.")

df_channels = df_channels.merge(channel_info, on="channel_id", how="left")
missing_channels = df_channels[df_channels["uploads_playlist_id"].isna()]
if len(missing_channels):
    print("Channel tidak dikembalikan API:", missing_channels["channel_name"].tolist())
df_channels = df_channels.dropna(subset=["uploads_playlist_id"]).reset_index(drop=True)

# Pilih metode pencarian video berdasarkan jumlah video channel.
df_channels["method"] = df_channels["channel_video_count"].map(
    lambda n: "search" if n > PLAYLIST_LIMIT else "playlist")

display(df_channels[["record_id", "channel_name", "category", "channel_video_count", "method"]])
print(df_channels["method"].value_counts().to_dict())

## 1.8. Fungsi Pencarian Video melalui Uploads Playlist

Mendefinisikan fungsi `search_uploads_playlist()` untuk channel dengan metode `playlist`. Fungsi ini:

* Membaca uploads playlist memakai endpoint `playlistItems.list` (part `snippet,contentDetails`, `maxResults=50`) halaman demi halaman dengan `nextPageToken`.
* Memeriksa setiap video: **tanggal terbit** berada dalam periode dan **judul/deskripsi** cocok dengan `MBG_PATTERN`. Hanya video yang lolos yang disimpan sebagai kandidat, sehingga kuota tidak terbuang untuk mengambil detail video yang jelas tidak relevan.
* **Berhenti lebih awal.** Karena playlist diurutkan dari video terbaru, jika **seluruh video pada satu halaman** sudah lebih lama dari `PERIOD_START`, halaman berikutnya pasti lebih lama lagi, sehingga penelusuran channel tersebut dihentikan.
* Menyimpan progres (token halaman berikutnya dan daftar kandidat) ke `state/playlist_<channel_id>.json` setiap selesai satu halaman, sehingga dapat dilanjutkan jika terhenti.
* Mencatat `MBG_PATTERN` yang dipakai di file state. Jika pola diubah, playlist channel tersebut otomatis ditelusuri ulang dengan pola baru, karena penyaringan dilakukan saat penelusuran.

In [ ]:
def matches_mbg(title, description):
    return bool(MBG_REGEX.search(f"{title or ''} {description or ''}"))


def search_uploads_playlist(channel):
    state_path = STATE_DIR / f"playlist_{channel['channel_id']}.json"
    new_state = {"pattern": MBG_PATTERN, "next_page_token": None, "pages": 0, "done": False, "video_ids": []}
    state = load_json(state_path, new_state)
    if state.get("pattern") != MBG_PATTERN:
        state = new_state  # Pola MBG berubah: telusuri ulang playlist dengan pola baru.

    while not state["done"]:
        data = youtube_get("playlistItems", {
            "part": "snippet,contentDetails",
            "playlistId": channel["uploads_playlist_id"],
            "maxResults": 50,
            "pageToken": state["next_page_token"],
        })
        page_dates = []
        for item in data.get("items", []):
            video_id = item["contentDetails"]["videoId"]
            published = pd.to_datetime(item["contentDetails"].get("videoPublishedAt"), utc=True, errors="coerce")
            if pd.isna(published):
                continue  # Video privat/dihapus tidak memiliki tanggal terbit.
            page_dates.append(published)
            in_period = START_UTC <= published < END_UTC
            if in_period and matches_mbg(item["snippet"].get("title"), item["snippet"].get("description")):
                state["video_ids"].append(video_id)

        state["pages"] += 1
        state["next_page_token"] = data.get("nextPageToken")
        # Berhenti jika halaman terakhir atau seluruh video di halaman ini sudah sebelum periode.
        all_before_period = bool(page_dates) and max(page_dates) < START_UTC
        state["done"] = state["next_page_token"] is None or all_before_period
        save_json(state_path, state)  # Simpan progres setiap halaman.
    return state

## 1.9. Fungsi Pencarian Video melalui Search API

Mendefinisikan fungsi `search_by_keyword()` untuk channel dengan metode `search`. Endpoint `search.list` dipanggil dengan parameter:

| Parameter | Nilai | Alasan |
|---|---|---|
| `channelId` | ID channel | hanya video dari channel terpilih |
| `q` | `MBG\|"makan bergizi gratis"` | keyword MBG (operator `\|` = ATAU) |
| `type` | `video` | tidak mengambil playlist/channel |
| `publishedAfter`, `publishedBefore` | batas satu kuartal | membagi periode menjadi rentang waktu (*window*) |
| `order` | `date` | urutan stabil berdasarkan tanggal |
| `maxResults` | `50` | jumlah maksimum per halaman |

**Batas 500 hasil.** Search API hanya mengembalikan maksimal 500 video (10 halaman) untuk satu pencarian per channel. Setelah itu API berhenti memberi `nextPageToken`, walaupun sebenarnya masih ada video lain. Karena itu:

* periode dibagi per **kuartal** oleh fungsi `make_quarter_windows()`;
* jika satu *window* mencapai 10 halaman, *window* tersebut dianggap terpotong (`capped`), lalu fungsi `split_window()` **membelahnya menjadi dua** dan keduanya dicari ulang, sampai setiap *window* berada di bawah batas.

Hasil Search API bergantung pada indeks pencarian YouTube, sehingga setiap kandidat tetap divalidasi ulang dengan `MBG_PATTERN` yang sama pada bagian 1.13. Dengan begitu, kriteria inklusi kedua metode tetap identik. Progres disimpan ke `state/search_<channel_id>.json`.

In [ ]:
def make_quarter_windows():
    # Bagi periode penelitian menjadi rentang waktu per kuartal kalender.
    bounds = [START_UTC]
    quarter_starts = pd.date_range(START_UTC, END_UTC, freq="QS")
    bounds += [t for t in quarter_starts if START_UTC < t < END_UTC]
    bounds.append(END_UTC)
    return [{"start": a.isoformat(), "end": b.isoformat(), "next_page_token": None,
             "pages": 0, "done": False, "capped": False}
            for a, b in zip(bounds[:-1], bounds[1:])]


def split_window(window):
    start, end = pd.Timestamp(window["start"]), pd.Timestamp(window["end"])
    if end - start <= pd.Timedelta(days=1):
        return []  # Window satu hari tidak dibelah lagi; tetap tercatat sebagai capped.
    middle = (start + (end - start) / 2).floor("s")
    return [{"start": a.isoformat(), "end": b.isoformat(), "next_page_token": None,
             "pages": 0, "done": False, "capped": False}
            for a, b in [(start, middle), (middle, end)]]


def to_rfc3339(timestamp_text):
    return pd.Timestamp(timestamp_text).strftime("%Y-%m-%dT%H:%M:%SZ")


def search_by_keyword(channel):
    state_path = STATE_DIR / f"search_{channel['channel_id']}.json"
    state = load_json(state_path, {"windows": make_quarter_windows(), "video_ids": []})

    i = 0
    while i < len(state["windows"]):  # Daftar window bisa bertambah ketika ada window yang dibelah.
        window = state["windows"][i]
        while not window["done"]:
            data = youtube_get("search", {
                "part": "id",
                "channelId": channel["channel_id"],
                "q": SEARCH_QUERY,
                "type": "video",
                "order": "date",
                "publishedAfter": to_rfc3339(window["start"]),
                "publishedBefore": to_rfc3339(window["end"]),
                "maxResults": 50,
                "pageToken": window["next_page_token"],
            })
            for item in data.get("items", []):
                if item.get("id", {}).get("videoId"):
                    state["video_ids"].append(item["id"]["videoId"])

            window["pages"] += 1
            window["next_page_token"] = data.get("nextPageToken")
            if window["pages"] >= 10:
                # Halaman ke-10 = 500 hasil (batas API): belah window menjadi dua lalu cari ulang.
                window.update(done=True, capped=True)
                state["windows"].extend(split_window(window))
            elif window["next_page_token"] is None:
                window["done"] = True
            save_json(state_path, state)  # Simpan progres setiap halaman.
        i += 1
    return state

## 1.10. Menjalankan Pencarian Video

Menjalankan pencarian video untuk semua channel sesuai metodenya masing-masing:

* Urutan: channel `playlist` terlebih dahulu, lalu channel `search` **dari jumlah video terkecil**. Jika kuota search terbatas, lebih banyak channel yang selesai, dan channel raksasa (Tribunnews, METRO TV, dll.) dikerjakan paling akhir.
* Jika **kuota unit** habis, seluruh pengambilan data pada sesi ini dihentikan.
* Jika hanya **kuota search** yang habis, channel dengan metode `playlist` dan pengambilan detail video tetap berjalan; sisa *window* search dilanjutkan pada sesi berikutnya.
* Error pada satu channel dicatat di `state/search_errors.json` dan channel tersebut dicoba lagi pada sesi berikutnya, tanpa menghentikan channel lain.

Setelah itu ditampilkan tabel progres pencarian per channel.

In [ ]:
unit_quota_exhausted = False
search_quota_exhausted = False
errors_path = STATE_DIR / "search_errors.json"
search_errors = load_json(errors_path, {})  # channel_id -> alasan error terakhir


def run_video_search():
    global unit_quota_exhausted, search_quota_exhausted
    # Channel playlist dulu, lalu channel search dari jumlah video terkecil,
    # agar lebih banyak channel selesai jika kuota search terbatas.
    ordered_channels = df_channels.sort_values(["method", "channel_video_count"])
    for channel in ordered_channels.to_dict("records"):
        if unit_quota_exhausted:
            break
        if channel["method"] == "search" and search_quota_exhausted:
            continue
        search_function = search_by_keyword if channel["method"] == "search" else search_uploads_playlist
        try:
            state = search_function(channel)
            search_errors.pop(channel["channel_id"], None)
            print(f"{channel['channel_name']:<40} {channel['method']:<8} kandidat: {len(set(state['video_ids']))}")
        except QuotaExceeded as e:
            print(f"{channel['channel_name']:<40} BERHENTI: kuota {e.quota_type} habis, lanjutkan sesi berikutnya.")
            if e.quota_type == "unit":
                unit_quota_exhausted = True
            else:
                search_quota_exhausted = True
        except ApiError as e:
            search_errors[channel["channel_id"]] = e.reason
            print(f"{channel['channel_name']:<40} ERROR: {e}")
        save_json(errors_path, search_errors)


if COLLECT_DATA:
    run_video_search()

In [ ]:
def is_unsplittable(window):
    # Window satu hari yang tetap mencapai 500 hasil tidak bisa dibelah lagi.
    duration = pd.Timestamp(window["end"]) - pd.Timestamp(window["start"])
    return window["capped"] and duration <= pd.Timedelta(days=1)


def get_search_progress():
    # Ringkas progres pencarian semua channel dari file state.
    rows = []
    for channel in df_channels.to_dict("records"):
        state = load_json(STATE_DIR / f"{channel['method']}_{channel['channel_id']}.json", None)
        if state is None:
            done, progress = False, "belum dimulai"
        elif channel["method"] == "playlist":
            done, progress = state["done"], f"{state['pages']} halaman"
        else:
            windows = state["windows"]
            done = all(w["done"] for w in windows)
            progress = f"{sum(w['done'] for w in windows)}/{len(windows)} window"
            unsplittable = sum(is_unsplittable(w) for w in windows)
            if unsplittable:
                progress += f", {unsplittable} window tetap terpotong"
        rows.append({"channel_id": channel["channel_id"], "channel_name": channel["channel_name"],
                     "method": channel["method"], "search_done": done, "progress": progress,
                     "error": search_errors.get(channel["channel_id"], ""),
                     "candidates": len(set((state or {}).get("video_ids", [])))})
    return pd.DataFrame(rows)


df_search_progress = get_search_progress()
display(df_search_progress)
print("Channel selesai dicari:", int(df_search_progress["search_done"].sum()), "/", len(df_search_progress))

## 1.11. Deduplikasi Kandidat Video

Menggabungkan kandidat video dari semua channel, lalu menghapus duplikasi berdasarkan `video_id`. Duplikasi dapat terjadi, misalnya, ketika video yang sama muncul di dua *window* search yang berdekatan setelah pembelahan.

In [ ]:
candidate_rows = []
for channel in df_channels.to_dict("records"):
    state = load_json(STATE_DIR / f"{channel['method']}_{channel['channel_id']}.json", {"video_ids": []})
    for video_id in state["video_ids"]:
        candidate_rows.append({"video_id": video_id, "search_method": channel["method"]})

df_candidates = pd.DataFrame(candidate_rows, columns=["video_id", "search_method"])
n_before = len(df_candidates)
df_candidates = df_candidates.drop_duplicates("video_id").reset_index(drop=True)
print(f"Kandidat: {n_before} baris -> {len(df_candidates)} video unik ({n_before - len(df_candidates)} duplikat dihapus)")

## 1.12. Fungsi Pengambilan Detail Video (Views, Likes, Jumlah Comment)

Mendefinisikan fungsi `get_video_details()` untuk mengambil detail setiap kandidat video melalui endpoint `videos.list`. Prosesnya dilakukan per **batch 50 `video_id`** (batas maksimum API), sehingga 1.000 video hanya memerlukan 20 request. Untuk setiap video dikumpulkan:

* `snippet`: judul, deskripsi, channel, waktu terbit, bahasa audio;
* `statistics`: jumlah view, like, dan comment;
* `contentDetails`: durasi dan ketersediaan *caption*;
* `liveStreamingDetails`: penanda bahwa video berasal dari siaran langsung (siaran langsung sering membahas banyak topik sekaligus).

Detail yang sudah diambil disimpan di `state/video_details.csv`, sehingga pada sesi berikutnya hanya ID baru yang diminta. ID yang tidak dikembalikan API (video dihapus/privat) dicatat di `state/unavailable_videos.json`.

In [ ]:
def get_video_details(video_ids):
    data = youtube_get("videos", {
        "part": "snippet,statistics,contentDetails,liveStreamingDetails",
        "id": ",".join(video_ids),
    })
    rows = []
    for item in data.get("items", []):
        snippet, stats, details = item["snippet"], item.get("statistics", {}), item.get("contentDetails", {})
        rows.append({
            "video_id": item["id"],
            "channel_id": snippet["channelId"],
            "channel_title": snippet["channelTitle"],
            "title": snippet["title"],
            "description": snippet.get("description", ""),
            "published_at": snippet["publishedAt"],
            "default_audio_language": snippet.get("defaultAudioLanguage", ""),
            "duration": details.get("duration", ""),
            "caption_available": details.get("caption", ""),
            "was_live": "liveStreamingDetails" in item,
            "view_count": stats.get("viewCount", ""),
            "like_count": stats.get("likeCount", ""),
            "comment_count": stats.get("commentCount", ""),
            "collected_at_utc": now_utc(),
        })
    return pd.DataFrame(rows)

In [ ]:
details_path = STATE_DIR / "video_details.csv"
unavailable_path = STATE_DIR / "unavailable_videos.json"
df_details = load_csv(details_path)
unavailable_videos = load_json(unavailable_path, [])

if COLLECT_DATA and not unit_quota_exhausted:
    already_fetched = set(df_details.get("video_id", [])) | set(unavailable_videos)
    pending_ids = [v for v in df_candidates["video_id"] if v not in already_fetched]
    print("Video yang perlu diambil detailnya:", len(pending_ids))
    try:
        for start in range(0, len(pending_ids), 50):
            batch = pending_ids[start:start + 50]
            result = get_video_details(batch)
            df_details = pd.concat([df_details, result.astype(str)], ignore_index=True)
            unavailable_videos += sorted(set(batch) - set(result["video_id"]))
            save_csv(df_details, details_path)  # Simpan setiap batch.
            save_json(unavailable_path, unavailable_videos)
    except QuotaExceeded:
        unit_quota_exhausted = True
        print("Kuota habis saat mengambil detail video; dilanjutkan pada sesi berikutnya.")

print("Detail video tersimpan:", len(df_details), "| tidak tersedia:", len(unavailable_videos))

## 1.13. Validasi Kriteria Video

Memeriksa ulang detail video dari API dengan **kriteria inklusi** yang sama untuk kedua metode pencarian:

1. video berasal dari salah satu dari 28 channel terpilih;
2. tanggal terbit berada dalam periode penelitian;
3. judul **atau** deskripsi cocok dengan `MBG_PATTERN`.

Video yang tidak lolos tidak dihapus dari data mentah, tetapi disimpan di `data/interim/youtube/excluded_videos.csv` beserta alasannya (`exclusion_reason`). Kolom `mbg_match_field` mencatat lokasi keyword ditemukan (`title` atau `description`). Video yang hanya cocok di deskripsi (misalnya siaran langsung dengan banyak segmen) tetap dipertahankan, tetapi dapat dipisahkan pada tahap analisis.

In [ ]:
def get_exclusion_reason(row):
    published = pd.to_datetime(row["published_at"], utc=True, errors="coerce")
    if row["channel_id"] not in set(df_channels["channel_id"]):
        return "not_selected_channel"
    if pd.isna(published) or not (START_UTC <= published < END_UTC):
        return "outside_period"
    if not matches_mbg(row["title"], row["description"]):
        return "no_mbg_keyword"
    return ""


if len(df_details):
    df_details["exclusion_reason"] = df_details.apply(get_exclusion_reason, axis=1)
    df_videos = df_details[df_details["exclusion_reason"] == ""].drop(columns="exclusion_reason").copy()
    df_excluded = df_details[df_details["exclusion_reason"] != ""]
else:
    df_videos, df_excluded = df_details.copy(), df_details.copy()

if len(df_videos):
    df_videos["mbg_match_field"] = df_videos["title"].map(
        lambda title: "title" if MBG_REGEX.search(title) else "description")
    df_videos = df_videos.merge(df_candidates, on="video_id", how="left")
save_csv(df_excluded, OUTPUT_DIR / "excluded_videos.csv")

print("Video memenuhi kriteria:", len(df_videos))
if len(df_excluded):
    print("Dikecualikan:", df_excluded["exclusion_reason"].value_counts().to_dict())

## 1.14. Status Pengumpulan Data

Memeriksa apakah pengumpulan sudah lengkap: semua channel selesai dicari dan semua kandidat sudah diambil
detailnya. Jika belum lengkap, jalankan ulang seluruh notebook setelah kuota harian di-*reset* (pukul 00.00 waktu
Pasifik, sekitar pukul 14.00–15.00 WIB).

In [ ]:
search_complete = bool(df_search_progress["search_done"].all()) if len(df_search_progress) else False
details_complete = set(df_candidates["video_id"]) <= (set(df_details.get("video_id", [])) | set(unavailable_videos))
COLLECTION_COMPLETE = search_complete and details_complete

print("Pencarian video semua channel :", search_complete)
print("Detail video                  :", details_complete)
print("Pemakaian sesi ini            :", usage)
print()
print("STATUS:", "LENGKAP" if COLLECTION_COMPLETE else
      "BELUM LENGKAP — jalankan ulang notebook pada sesi/hari berikutnya.")

# 2. Preprocessing Data

Pembersihan pada tahap ini bersifat **umum**: teks asli selalu dipertahankan, dan kolom bersih ditambahkan di
sebelahnya.

| Langkah | Alasan |
|---|---|
| Hapus `video_id` kosong dan duplikat | Satu video hanya muncul sekali |
| `title_clean`, `description_clean`: decode entitas HTML (`&amp;` → `&`), normalisasi Unicode NFC, rapikan spasi | Teks konsisten tanpa mengubah makna |
| Konversi tipe data: tanggal → datetime UTC, angka → integer | Siap dianalisis |
| Ekstraksi lokasi: `location_names`, `province`, `location_match_type`, `location_source` | Bahan analisis wilayah; teks asli tidak diubah |

Yang **sengaja tidak dilakukan**: *lowercase*, hapus *stopword*, *stemming*, hapus emoji/URL/tanda baca.
Kolom `title` dan `description` asli selalu dipertahankan.

## 2.1. Fungsi Pembersihan Teks

`clean_text()`: decode entitas HTML, normalisasi Unicode NFC (agar karakter yang tampak sama memiliki kode yang sama), dan merapikan spasi/baris baru menjadi satu spasi.

In [ ]:
def clean_text(text):
    text = html.unescape(str(text))
    text = unicodedata.normalize("NFC", text)
    return re.sub(r"\s+", " ", text).strip()

## 2.2. Pembersihan Dataset Video

Membersihkan dataset video: menghapus `video_id` kosong/duplikat, menambahkan informasi channel (`record_id`, `channel_name`, `category`), mengonversi tanggal ke datetime UTC, mengonversi kolom angka ke integer (nilai kosong tetap kosong, bukan nol), serta membuat `title_clean` dan `description_clean`.

In [ ]:
cleaning_log = {}

videos = df_videos.copy()
cleaning_log["video_rows_before"] = len(videos)
if len(videos):
    videos["video_id"] = videos["video_id"].str.strip()
    cleaning_log["video_empty_id"] = int((videos["video_id"] == "").sum())
    videos = videos[videos["video_id"] != ""]
    cleaning_log["video_duplicate_id"] = int(videos["video_id"].duplicated().sum())
    videos = videos.drop_duplicates("video_id")

    videos = videos.merge(df_channels[["channel_id", "record_id", "channel_name", "category"]],
                          on="channel_id", how="left")
    videos["published_at"] = pd.to_datetime(videos["published_at"], utc=True, errors="coerce", format="ISO8601")
    videos["collected_at_utc"] = pd.to_datetime(videos["collected_at_utc"], utc=True, errors="coerce", format="ISO8601")
    for col in ["view_count", "like_count", "comment_count"]:
        videos[col] = pd.to_numeric(videos[col], errors="coerce").astype("Int64")
    videos["was_live"] = videos["was_live"].astype(str).eq("True")
    videos["title_clean"] = videos["title"].map(clean_text)
    videos["description_clean"] = videos["description"].map(clean_text)
videos = videos.reset_index(drop=True)
cleaning_log["video_rows_after"] = len(videos)

print("Dataset video:", videos.shape)

## 2.3. Ekstraksi Lokasi

Dashboard membutuhkan informasi **wilayah** yang diberitakan. YouTube API tidak menyediakan lokasi kejadian, sehingga lokasi diambil dari **nama wilayah yang disebut di teks** dengan kamus wilayah administratif Indonesia (38 provinsi beserta kabupaten/kotanya).

| Sumber | Cara | Keterangan |
|---|---|---|
| Judul video | Dicocokkan lebih dulu | Judul berita biasanya menyebut lokasi kejadian |
| Deskripsi video | Dipakai jika judul tidak menyebut lokasi | Dateline Jakarta ("JAKARTA, KOMPAS.TV -") dan alamat redaksi di footer dibuang karena menandakan kantor redaksi; dateline daerah ("GARUT, KOMPAS.TV -") dipertahankan |

Aturan pencocokan:

* Teks dinormalisasi (huruf kecil, tanda baca menjadi spasi), lalu nama terpanjang dicocokkan lebih dulu, misalnya "Bandung Barat" sebelum "Bandung".
* **Alias dan singkatan**: Jabar, Jateng, Jogja, Solo, Tasik, jkt, bdg, dan lain-lain.
* **Nama ambigu** yang juga kata sehari-hari (misalnya *Pati, Malang, Medan, Padang, Solo*) hanya dihitung jika didahului kata konteks seperti "di", "ke", "dari", "kota", atau "kabupaten". Untuk *Serang* dan *Batu*, kata konteksnya harus kata administratif ("Kota Serang", "Kota Batu"), karena "di serang" juga berarti "diserang".
* **Ejaan alternatif dan salah ketik yang umum** (misalnya "tanggerang", "makasar", "jogya", "jawatimur") dimasukkan sebagai alias. Pencocokan typo otomatis (kata yang berbeda 1 huruf dari nama wilayah) sudah diuji pada data, tetapi hasilnya sebagian besar salah. Contohnya "di maling" terbaca Malang, "di banding" terbaca Bandung, dan "asal asalan" terbaca Asahan, sehingga cara ini tidak dipakai.
* **Ungkapan** yang memuat nama wilayah tetapi tidak merujuk wilayah tersebut, misalnya "dari Sabang sampai Merauke" dan "padang mahsyar", dibuang sebelum pencocokan.
* Nama media daerah seperti "Tribun Jateng" atau "Radar Bogor" dibuang dari teks video karena merupakan nama media, bukan lokasi kejadian.

Kolom yang ditambahkan:

* `location_names`: nama wilayah yang disebut (dipisah `; `);
* `province`: provinsi dari wilayah tersebut;
* `location_match_type`: `exact` (nama resmi) atau `alias` (singkatan/nama populer/ejaan alternatif);
* `location_source` (khusus video): `title`, `description`, atau `none`.

Lokasi yang ditemukan adalah lokasi yang **disebut**, belum tentu lokasi kejadian. Contoh acak hasil ekstraksi ditampilkan untuk pemeriksaan manual (presisi).

### 2.3.1 Kamus Wilayah Indonesia

Kamus disusun dari daftar wilayah administratif Kemendagri: provinsi → kabupaten/kota. Nama ditulis tanpa awalan "Kabupaten"/"Kota"; jika kabupaten dan kota bernama sama (misalnya Kabupaten Bogor dan Kota Bogor), nama cukup ditulis sekali.

In [ ]:
REGIONS = {
    "Aceh": ["Simeulue", "Aceh Singkil", "Aceh Selatan", "Aceh Tenggara", "Aceh Timur", "Aceh Tengah",
             "Aceh Barat", "Aceh Besar", "Pidie", "Bireuen", "Aceh Utara", "Aceh Barat Daya", "Gayo Lues",
             "Aceh Tamiang", "Nagan Raya", "Aceh Jaya", "Bener Meriah", "Pidie Jaya", "Banda Aceh", "Sabang",
             "Langsa", "Lhokseumawe", "Subulussalam"],
    "Sumatera Utara": ["Nias", "Mandailing Natal", "Tapanuli Selatan", "Tapanuli Tengah", "Tapanuli Utara", "Toba",
                       "Labuhanbatu", "Asahan", "Simalungun", "Dairi", "Karo", "Deli Serdang", "Langkat",
                       "Nias Selatan", "Humbang Hasundutan", "Pakpak Bharat", "Samosir", "Serdang Bedagai",
                       "Batu Bara", "Padang Lawas Utara", "Padang Lawas", "Labuhanbatu Selatan",
                       "Labuhanbatu Utara", "Nias Utara", "Nias Barat", "Sibolga", "Tanjungbalai",
                       "Pematangsiantar", "Tebing Tinggi", "Medan", "Binjai", "Padangsidimpuan", "Gunungsitoli"],
    "Sumatera Barat": ["Kepulauan Mentawai", "Pesisir Selatan", "Solok", "Sijunjung", "Tanah Datar",
                       "Padang Pariaman", "Agam", "Lima Puluh Kota", "Pasaman", "Solok Selatan", "Dharmasraya",
                       "Pasaman Barat", "Padang", "Sawahlunto", "Padang Panjang", "Bukittinggi", "Payakumbuh",
                       "Pariaman"],
    "Riau": ["Kuantan Singingi", "Indragiri Hulu", "Indragiri Hilir", "Pelalawan", "Siak", "Kampar", "Rokan Hulu",
             "Bengkalis", "Rokan Hilir", "Kepulauan Meranti", "Pekanbaru", "Dumai"],
    "Kepulauan Riau": ["Karimun", "Bintan", "Natuna", "Lingga", "Kepulauan Anambas", "Batam", "Tanjungpinang"],
    "Jambi": ["Kerinci", "Merangin", "Sarolangun", "Batanghari", "Muaro Jambi", "Tanjung Jabung Timur",
              "Tanjung Jabung Barat", "Tebo", "Bungo", "Jambi", "Sungai Penuh"],
    "Sumatera Selatan": ["Ogan Komering Ulu", "Ogan Komering Ilir", "Muara Enim", "Lahat", "Musi Rawas",
                         "Musi Banyuasin", "Banyuasin", "Ogan Komering Ulu Selatan", "Ogan Komering Ulu Timur",
                         "Ogan Ilir", "Empat Lawang", "Penukal Abab Lematang Ilir", "Musi Rawas Utara",
                         "Palembang", "Prabumulih", "Pagar Alam", "Lubuklinggau"],
    "Kepulauan Bangka Belitung": ["Bangka", "Belitung", "Bangka Barat", "Bangka Tengah", "Bangka Selatan",
                                  "Belitung Timur", "Pangkalpinang"],
    "Bengkulu": ["Bengkulu Selatan", "Rejang Lebong", "Bengkulu Utara", "Kaur", "Seluma", "Mukomuko", "Lebong",
                 "Kepahiang", "Bengkulu Tengah", "Bengkulu"],
    "Lampung": ["Lampung Barat", "Tanggamus", "Lampung Selatan", "Lampung Timur", "Lampung Tengah",
                "Lampung Utara", "Way Kanan", "Tulang Bawang", "Pesawaran", "Pringsewu", "Mesuji",
                "Tulang Bawang Barat", "Pesisir Barat", "Bandar Lampung",
                "Kota Metro"],  # ditulis lengkap agar tidak tertukar dengan "Metro TV"
    "DKI Jakarta": ["Jakarta", "Jakarta Pusat", "Jakarta Utara", "Jakarta Barat", "Jakarta Selatan",
                    "Jakarta Timur", "Kepulauan Seribu"],
    "Jawa Barat": ["Bogor", "Sukabumi", "Cianjur", "Bandung", "Garut", "Tasikmalaya", "Ciamis", "Kuningan",
                   "Cirebon", "Majalengka", "Sumedang", "Indramayu", "Subang", "Purwakarta", "Karawang", "Bekasi",
                   "Bandung Barat", "Pangandaran", "Depok", "Cimahi",
                   "Kota Banjar"],  # ditulis lengkap; "Banjar" saja dipetakan ke Kabupaten Banjar, Kalimantan Selatan
    "Banten": ["Pandeglang", "Lebak", "Tangerang", "Serang", "Cilegon", "Tangerang Selatan"],
    "Jawa Tengah": ["Cilacap", "Banyumas", "Purbalingga", "Banjarnegara", "Kebumen", "Purworejo", "Wonosobo",
                    "Magelang", "Boyolali", "Klaten", "Sukoharjo", "Wonogiri", "Karanganyar", "Sragen", "Grobogan",
                    "Blora", "Rembang", "Pati", "Kudus", "Jepara", "Demak", "Semarang", "Temanggung", "Kendal",
                    "Batang", "Pekalongan", "Pemalang", "Tegal", "Brebes", "Surakarta", "Salatiga"],
    "Daerah Istimewa Yogyakarta": ["Kulon Progo", "Bantul", "Gunungkidul", "Sleman", "Yogyakarta"],
    "Jawa Timur": ["Pacitan", "Ponorogo", "Trenggalek", "Tulungagung", "Blitar", "Kediri", "Malang", "Lumajang",
                   "Jember", "Banyuwangi", "Bondowoso", "Situbondo", "Probolinggo", "Pasuruan", "Sidoarjo",
                   "Mojokerto", "Jombang", "Nganjuk", "Madiun", "Magetan", "Ngawi", "Bojonegoro", "Tuban",
                   "Lamongan", "Gresik", "Bangkalan", "Sampang", "Pamekasan", "Sumenep", "Surabaya", "Batu"],
    "Bali": ["Jembrana", "Tabanan", "Badung", "Gianyar", "Klungkung", "Bangli", "Karangasem", "Buleleng",
             "Denpasar"],
    "Nusa Tenggara Barat": ["Lombok Barat", "Lombok Tengah", "Lombok Timur", "Lombok Utara", "Sumbawa",
                            "Sumbawa Barat", "Dompu", "Bima", "Mataram"],
    "Nusa Tenggara Timur": ["Kupang", "Timor Tengah Selatan", "Timor Tengah Utara", "Belu", "Malaka", "Alor",
                            "Lembata", "Flores Timur", "Sikka", "Ende", "Ngada", "Nagekeo", "Manggarai",
                            "Manggarai Barat", "Manggarai Timur", "Sumba Barat", "Sumba Timur", "Sumba Tengah",
                            "Sumba Barat Daya", "Rote Ndao", "Sabu Raijua"],
    "Kalimantan Barat": ["Sambas", "Bengkayang", "Landak", "Mempawah", "Sanggau", "Ketapang", "Sintang",
                         "Kapuas Hulu", "Sekadau", "Melawi", "Kayong Utara", "Kubu Raya", "Pontianak", "Singkawang"],
    "Kalimantan Tengah": ["Kotawaringin Barat", "Kotawaringin Timur", "Kapuas", "Barito Selatan", "Barito Utara",
                          "Barito Timur", "Sukamara", "Lamandau", "Seruyan", "Katingan", "Pulang Pisau",
                          "Gunung Mas", "Murung Raya", "Palangka Raya"],
    "Kalimantan Selatan": ["Tanah Laut", "Kotabaru", "Banjar", "Barito Kuala", "Tapin", "Hulu Sungai Selatan",
                           "Hulu Sungai Tengah", "Hulu Sungai Utara", "Tabalong", "Tanah Bumbu", "Balangan",
                           "Banjarmasin", "Banjarbaru"],
    "Kalimantan Timur": ["Paser", "Kutai Barat", "Kutai Kartanegara", "Kutai Timur", "Berau",
                         "Penajam Paser Utara", "Mahakam Ulu", "Balikpapan", "Samarinda", "Bontang"],
    "Kalimantan Utara": ["Malinau", "Bulungan", "Tana Tidung", "Nunukan", "Tarakan"],
    "Sulawesi Utara": ["Bolaang Mongondow", "Minahasa", "Kepulauan Sangihe", "Kepulauan Talaud", "Minahasa Selatan",
                       "Minahasa Utara", "Minahasa Tenggara", "Bolaang Mongondow Utara",
                       "Bolaang Mongondow Selatan", "Bolaang Mongondow Timur", "Kepulauan Siau Tagulandang Biaro",
                       "Manado", "Bitung", "Tomohon", "Kotamobagu"],
    "Gorontalo": ["Boalemo", "Gorontalo", "Pohuwato", "Bone Bolango", "Gorontalo Utara"],
    "Sulawesi Tengah": ["Banggai Kepulauan", "Banggai", "Banggai Laut", "Morowali", "Morowali Utara", "Poso",
                        "Donggala", "Tolitoli", "Buol", "Parigi Moutong", "Tojo Una-Una", "Sigi", "Palu"],
    "Sulawesi Barat": ["Majene", "Polewali Mandar", "Mamasa", "Mamuju", "Mamuju Tengah", "Pasangkayu"],
    "Sulawesi Selatan": ["Kepulauan Selayar", "Bulukumba", "Bantaeng", "Jeneponto", "Takalar", "Gowa", "Sinjai",
                         "Maros", "Pangkajene dan Kepulauan", "Barru", "Bone", "Soppeng", "Wajo",
                         "Sidenreng Rappang", "Pinrang", "Enrekang", "Luwu", "Luwu Utara", "Luwu Timur",
                         "Tana Toraja", "Toraja Utara", "Makassar", "Parepare", "Palopo"],
    "Sulawesi Tenggara": ["Buton", "Buton Utara", "Buton Tengah", "Buton Selatan", "Muna", "Muna Barat", "Konawe",
                          "Konawe Selatan", "Konawe Utara", "Konawe Kepulauan", "Kolaka", "Kolaka Utara",
                          "Kolaka Timur", "Bombana", "Wakatobi", "Kendari", "Baubau"],
    "Maluku": ["Kepulauan Tanimbar", "Maluku Tenggara", "Maluku Tengah", "Buru", "Buru Selatan", "Kepulauan Aru",
               "Seram Bagian Barat", "Seram Bagian Timur", "Maluku Barat Daya", "Ambon", "Tual"],
    "Maluku Utara": ["Halmahera Barat", "Halmahera Tengah", "Halmahera Selatan", "Halmahera Utara",
                     "Halmahera Timur", "Kepulauan Sula", "Pulau Morotai", "Pulau Taliabu", "Ternate",
                     "Tidore Kepulauan"],
    "Papua": ["Jayapura", "Kepulauan Yapen", "Biak Numfor", "Sarmi", "Keerom", "Waropen", "Supiori",
              "Mamberamo Raya"],
    "Papua Barat": ["Fakfak", "Kaimana", "Teluk Wondama", "Teluk Bintuni", "Manokwari", "Manokwari Selatan",
                    "Pegunungan Arfak"],
    "Papua Barat Daya": ["Sorong", "Sorong Selatan", "Raja Ampat", "Tambrauw", "Maybrat"],
    "Papua Selatan": ["Merauke", "Boven Digoel", "Mappi", "Asmat"],
    # Kabupaten Puncak tidak dimasukkan: "Puncak" umumnya merujuk kawasan Puncak, Bogor.
    "Papua Tengah": ["Nabire", "Puncak Jaya", "Paniai", "Mimika", "Dogiyai", "Intan Jaya", "Deiyai"],
    "Papua Pegunungan": ["Jayawijaya", "Pegunungan Bintang", "Yahukimo", "Tolikara", "Mamberamo Tengah", "Yalimo",
                         "Lanny Jaya", "Nduga"],
}

print("Provinsi:", len(REGIONS), "| nama kabupaten/kota:", sum(len(names) for names in REGIONS.values()))

### 2.3.2 Alias, Nama Ambigu, dan Fungsi Pencocokan

* `ALIASES`: singkatan, nama populer, dan ejaan alternatif yang umum → nama resmi di kamus.
* `NEEDS_CONTEXT`: nama yang juga kata sehari-hari atau nama orang, misalnya "malang" (nasib malang), "medan" (medan perang), "solo" (konser solo), "biak" (berkembang biak), "samosir" (marga). Nama ini hanya dihitung jika didahului salah satu `CONTEXT_WORDS` ("di", "ke", "dari", "kota", "kabupaten", ...).
* `NEEDS_ADMIN_CONTEXT`: nama yang tetap ambigu walaupun didahului "di", misalnya "di serang" (= diserang) dan "dari batu" (lebih keras dari batu). Nama ini hanya dihitung jika didahului kata administratif seperti "kota" atau "kabupaten".
* `NOT_LOCATION_PHRASES`: ungkapan yang memuat nama wilayah tetapi tidak merujuk wilayah tersebut, misalnya "dari Sabang sampai Merauke" dan "padang mahsyar".
* `find_locations()`: mencari semua nama wilayah dalam teks dan mencatat jenis pencocokannya (`exact` atau `alias`).
* `summarize_locations()`: meringkas hasil pencocokan menjadi kolom `location_names`, `province`, dan `location_match_type`.

In [ ]:
ALIASES = {
    # Singkatan provinsi
    "jabar": "Jawa Barat", "jateng": "Jawa Tengah", "jatim": "Jawa Timur", "sumut": "Sumatera Utara",
    "sumbar": "Sumatera Barat", "sumsel": "Sumatera Selatan", "kalbar": "Kalimantan Barat",
    "kalteng": "Kalimantan Tengah", "kalsel": "Kalimantan Selatan", "kaltim": "Kalimantan Timur",
    "kaltara": "Kalimantan Utara", "sulut": "Sulawesi Utara", "sulteng": "Sulawesi Tengah",
    "sulsel": "Sulawesi Selatan", "sultra": "Sulawesi Tenggara", "sulbar": "Sulawesi Barat",
    "malut": "Maluku Utara", "kepri": "Kepulauan Riau", "babel": "Kepulauan Bangka Belitung",
    "bangka belitung": "Kepulauan Bangka Belitung", "ntb": "Nusa Tenggara Barat", "ntt": "Nusa Tenggara Timur",
    "diy": "Daerah Istimewa Yogyakarta", "dki": "DKI Jakarta",
    # Ejaan alternatif provinsi
    "sumatra utara": "Sumatera Utara", "sumatra barat": "Sumatera Barat", "sumatra selatan": "Sumatera Selatan",
    "jawabarat": "Jawa Barat", "jawatengah": "Jawa Tengah", "jawatimur": "Jawa Timur",
    # Pulau/kawasan yang hanya berada di satu provinsi
    "lombok": "Nusa Tenggara Barat", "madura": "Jawa Timur", "flores": "Nusa Tenggara Timur",
    "sumba": "Nusa Tenggara Timur",
    # Nama populer dan singkatan kota/kabupaten.
    # "sby", "smg", dan "mks" sengaja tidak dipakai: di teks lebih sering berarti SBY, "semoga", dan "makasih".
    "jogja": "Yogyakarta", "yogya": "Yogyakarta", "solo": "Surakarta", "tasik": "Tasikmalaya", "jkt": "Jakarta",
    "jakpus": "Jakarta Pusat", "jakut": "Jakarta Utara", "jakbar": "Jakarta Barat", "jaksel": "Jakarta Selatan",
    "jaktim": "Jakarta Timur", "tangsel": "Tangerang Selatan", "bdg": "Bandung", "purwokerto": "Banyumas",
    "cikarang": "Bekasi", "cibinong": "Bogor", "sampit": "Kotawaringin Timur", "timika": "Mimika",
    "wamena": "Jayawijaya", "biak": "Biak Numfor", "siantar": "Pematangsiantar", "toraja": "Tana Toraja",
    "sidrap": "Sidenreng Rappang", "pangkep": "Pangkajene dan Kepulauan", "polman": "Polewali Mandar",
    "selayar": "Kepulauan Selayar", "oku": "Ogan Komering Ulu", "oki": "Ogan Komering Ilir",
    "muba": "Musi Banyuasin", "kukar": "Kutai Kartanegara", "kutim": "Kutai Timur", "kubar": "Kutai Barat",
    "ppu": "Penajam Paser Utara", "kotim": "Kotawaringin Timur", "kobar": "Kotawaringin Barat",
    "mentawai": "Kepulauan Mentawai", "sangihe": "Kepulauan Sangihe", "talaud": "Kepulauan Talaud",
    "anambas": "Kepulauan Anambas", "morotai": "Pulau Morotai", "taliabu": "Pulau Taliabu",
    "tidore": "Tidore Kepulauan", "bintuni": "Teluk Bintuni", "tanimbar": "Kepulauan Tanimbar",
    "yapen": "Kepulauan Yapen",
    # Ejaan alternatif dan salah ketik yang umum
    "jogjakarta": "Yogyakarta", "jogya": "Yogyakarta", "tanggerang": "Tangerang", "makasar": "Makassar",
    "palangkaraya": "Palangka Raya", "pekan baru": "Pekanbaru", "bukit tinggi": "Bukittinggi",
    "bandarlampung": "Bandar Lampung", "pangkal pinang": "Pangkalpinang", "lubuk linggau": "Lubuklinggau",
    "pematang siantar": "Pematangsiantar", "tanjung balai": "Tanjungbalai", "tanjung pinang": "Tanjungpinang",
    "gunung sitoli": "Gunungsitoli", "padang sidempuan": "Padangsidimpuan", "bau bau": "Baubau",
    "pare pare": "Parepare", "gunung kidul": "Gunungkidul", "kulonprogo": "Kulon Progo",
    "labuhan batu": "Labuhanbatu", "toli toli": "Tolitoli", "muko muko": "Mukomuko", "fak fak": "Fakfak",
    "banjar baru": "Banjarbaru",
}

CONTEXT_WORDS = {"di", "ke", "dari", "asal", "sekitar", "kota", "kabupaten", "kab", "provinsi", "prov",
                 "wilayah", "daerah", "warga"}
ADMIN_WORDS = {"kota", "kabupaten", "kab", "provinsi", "prov", "wilayah", "daerah", "warga"}

# Nama yang juga kata sehari-hari atau nama orang: dihitung hanya jika didahului CONTEXT_WORDS.
NEEDS_CONTEXT = {
    "pati", "agam", "buru", "bima", "banjar", "landak", "lebak", "muna", "kudus", "batang", "padang", "solo",
    "tegal", "bone", "belu", "malaka", "sabang", "kaur", "lingga", "medan", "palu", "malang", "kuningan", "karo",
    "ende", "sorong", "lombok", "samosir", "ngada", "biak", "bangka", "lahat", "tebing tinggi", "tanah datar",
    "pesisir selatan", "pesisir barat", "sungai penuh", "gunung mas", "empat lawang", "pagar alam",
    "lima puluh kota",
}
# Tetap ambigu setelah "di"/"dari" ("di serang" = diserang, "dari batu", "batu bara", "bau bau korupsi"):
# hanya dihitung jika didahului ADMIN_WORDS.
NEEDS_ADMIN_CONTEXT = {"serang", "batu", "batu bara", "bau bau"}
# Ungkapan yang memuat nama wilayah tetapi tidak merujuk wilayah tersebut.
NOT_LOCATION_PHRASES = re.compile(r"\bsabang\s+(?:sampai\s+|hingga\s+|ke\s+|sd\s+)?merauke\b"
                                  r"|\bpadang\s+(?:mahsyar|mahsar|gurun|pasir|rumput)\b")


def normalize_location_text(text):
    # Huruf kecil dan tanda baca menjadi spasi: "Kab. Garut," -> "kab garut".
    return " ".join(re.findall(r"[a-z0-9]+", str(text).lower()))


# Peta bentuk ternormalisasi -> nama resmi, dan nama resmi -> provinsi.
LOCATION_LOOKUP = {}
LOCATION_PROVINCE = {}
for province, names in REGIONS.items():
    for name in [province] + names:
        LOCATION_LOOKUP[normalize_location_text(name)] = name
        # Jika ada nama yang sama di dua provinsi, nama itu tidak dipetakan ke provinsi mana pun.
        LOCATION_PROVINCE[name] = province if LOCATION_PROVINCE.get(name, province) == province else ""
for alias, name in ALIASES.items():
    assert name in LOCATION_PROVINCE, f"Alias {alias} menunjuk nama yang tidak ada di kamus: {name}"
    LOCATION_LOOKUP[alias] = name

FIRST_WORDS = {key.split()[0] for key in LOCATION_LOOKUP}
MAX_NAME_WORDS = max(len(key.split()) for key in LOCATION_LOOKUP)


def find_locations(text):
    # Hasil: daftar (nama_resmi, jenis_cocok) dengan jenis_cocok = exact / alias.
    words = NOT_LOCATION_PHRASES.sub(" ", normalize_location_text(text)).split()
    found = []
    i = 0
    while i < len(words):
        previous = words[i - 1] if i > 0 else ""
        n_matched = 1
        if words[i] in FIRST_WORDS:
            # Cocokkan nama terpanjang lebih dulu, misalnya "bandung barat" sebelum "bandung".
            for n in range(MAX_NAME_WORDS, 0, -1):
                key = " ".join(words[i:i + n])
                if key not in LOCATION_LOOKUP:
                    continue
                if key in NEEDS_CONTEXT and previous not in CONTEXT_WORDS:
                    continue
                if key in NEEDS_ADMIN_CONTEXT and previous not in ADMIN_WORDS:
                    continue
                found.append((LOCATION_LOOKUP[key], "alias" if key in ALIASES else "exact"))
                n_matched = n
                break
        i += n_matched
    return found


def summarize_locations(matches):
    names = list(dict.fromkeys(name for name, _ in matches))
    provinces = list(dict.fromkeys(LOCATION_PROVINCE[name] for name in names if LOCATION_PROVINCE[name]))
    match_type = "alias" if any(t == "alias" for _, t in matches) else ("exact" if matches else "")
    return "; ".join(names), "; ".join(provinces), match_type


print(summarize_locations(find_locations("Puluhan Siswa di Kab. Bandung Barat Keracunan, Jabar")))
print(summarize_locations(find_locations("MBG merata dari Sabang sampai Merauke, guru di serang buzzer")))
print(summarize_locations(find_locations("Keracunan di Kota Serang dan di Malang, nasib malang")))

### 2.3.3 Menerapkan Ekstraksi Lokasi pada Video

Judul dicocokkan lebih dulu; deskripsi hanya dipakai jika judul tidak menyebut lokasi. Proses ini tidak memanggil API.

In [ ]:
# Dateline Jakarta, misalnya "JAKARTA, KOMPAS.TV -", "Jakarta (ANTARA) -", "TEMPO.CO, Jakarta -".
# Dicari di seluruh deskripsi karena sebagian channel menaruh iklan sebelum dateline.
JAKARTA_DATELINE = re.compile(r"\bjakarta\s*(?:,|\()\s*[\w. ]{2,20}?\)?\s*[-–—]|\b[\w.]{2,20},\s*jakarta\s*[-–—]",
                              re.IGNORECASE)
# Alamat redaksi di footer deskripsi, misalnya "Jalan Cikini IV No. 11 Cikini, Jakarta Pusat 10350".
OFFICE_ADDRESS = re.compile(r"\b(?:jalan|jl|rt)\b.{0,150}?\b\d{5}\b", re.IGNORECASE)
# Nama media daerah, misalnya "Tribun Jateng", "TribunJatim", "Radar Bogor", "Pos Kupang".
MEDIA_NAMES = re.compile(r"\b(tribun|radar|pos)\s?[a-z]+", re.IGNORECASE)
LOCATION_COLUMNS = ["location_names", "province", "location_match_type"]


def locate_video(title, description):
    title_matches = find_locations(MEDIA_NAMES.sub(" ", title))
    if title_matches:
        return summarize_locations(title_matches) + ("title",)
    for pattern in [JAKARTA_DATELINE, OFFICE_ADDRESS, MEDIA_NAMES]:
        description = pattern.sub(" ", description)
    description_matches = find_locations(description)
    if description_matches:
        return summarize_locations(description_matches) + ("description",)
    return "", "", "", "none"


if len(videos):
    video_locations = [locate_video(title, description)
                       for title, description in zip(videos["title_clean"], videos["description_clean"])]
    videos[LOCATION_COLUMNS + ["location_source"]] = pd.DataFrame(video_locations, index=videos.index)
    print("Video yang menyebut lokasi  :", int((videos["location_names"] != "").sum()), "dari", len(videos),
          videos["location_source"].value_counts().to_dict())

Contoh acak hasil ekstraksi untuk pemeriksaan manual. Dari sampel seperti ini dapat dihitung presisi, yaitu persentase lokasi yang memang benar disebut sebagai wilayah.

In [ ]:
if len(videos) and (videos["location_names"] != "").any():
    video_sample = videos[videos["location_names"] != ""]
    display(video_sample.sample(min(10, len(video_sample)), random_state=42)[
        ["title", "location_names", "province", "location_source"]])

## 2.4. Ringkasan Pembersihan

Menampilkan jumlah baris yang dihapus pada setiap langkah, serta contoh hasil pembersihan.

In [ ]:
df_cleaning_log = pd.DataFrame(list(cleaning_log.items()), columns=["step", "count"])
display(df_cleaning_log)

if len(videos):
    display(videos[["video_id", "channel_name", "published_at", "title", "mbg_match_field",
                    "search_method", "view_count", "comment_count"]].head())

## 2.5. Penyimpanan Dataset

Menyimpan dataset bersih ke folder `data/source/`:

* `videos_clean.csv`: satu baris per video, beserta kolom lokasi (`location_*`, `province`);
* `cleaning_log.csv`: catatan pembersihan;
* `run_metadata.json`: konfigurasi, status, dan jumlah data, agar dataset dapat ditelusuri asal-usulnya.

In [ ]:
save_csv(videos, OUTPUT_DIR / "videos_clean.csv")
save_csv(df_cleaning_log, OUTPUT_DIR / "cleaning_log.csv")
print("videos_clean.csv:", len(videos), "baris")

run_metadata = {
    "created_at_utc": now_utc(),
    "video_period_wib": [PERIOD_START, PERIOD_END],
    "mbg_pattern": MBG_PATTERN,
    "search_query": SEARCH_QUERY,
    "n_channels": len(df_channels),
    "search_method_channels": df_channels.loc[df_channels["method"] == "search", "channel_name"].tolist(),
    "collection_complete": COLLECTION_COMPLETE,
    "counts": {"videos": len(videos),
               "videos_with_location": int((videos.get("location_names", pd.Series(dtype=str)) != "").sum())},
    "cleaning_log": cleaning_log,
    "library_versions": {"pandas": pd.__version__, "requests": requests.__version__},
}
save_json(OUTPUT_DIR / "run_metadata.json", run_metadata)
print(json.dumps(run_metadata, ensure_ascii=False, indent=2, default=str))

# 3. Exploratory Data Analysis

EDA dilakukan pada dataset video bersih untuk memahami struktur, kelengkapan, dan sebaran data. Frekuensi kata,
wordcloud, dan kemunculan keyword hanya memberi gambaran awal, bukan hasil *topic modeling*.

Fungsi `save_figure()` menyimpan setiap grafik ke `data/raw/youtube/figures/`.

In [ ]:
assert len(videos) > 0, "Dataset video masih kosong; jalankan pengumpulan data terlebih dahulu."


def save_figure(name):
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"{name}.png", dpi=150)
    plt.show()


# Stopword sederhana khusus untuk frekuensi kata pada EDA (dataset tidak diubah).
EDA_STOPWORDS = set('''
yang dan di ke dari ini itu untuk dengan pada adalah akan juga tidak ada atau karena oleh dalam
sudah saja bisa kita kami mereka dia ya aja jadi kalau kalo lagi masih belum harus buat sama
tapi tak nya pun lebih bukan apa siapa mana hanya agar seperti saat setelah tersebut namun para
the and of to in for on is with live news video berita tv com www https http youtube
mbg makan bergizi gratis program
'''.split())


def top_words(texts, n=20):
    words = []
    for text in texts:
        words += [w for w in re.findall(r"[a-z]{3,}", str(text).lower()) if w not in EDA_STOPWORDS]
    return pd.DataFrame(Counter(words).most_common(n), columns=["word", "frequency"])

## 3.1. Analisis Dataset Video

### 3.1.1 Informasi Struktur Dataset

Menampilkan jumlah baris dan kolom, tipe data setiap kolom, serta contoh beberapa baris dataset video.

In [ ]:
print("Jumlah baris, kolom:", videos.shape)
videos.info()
display(videos.head(3))

### 3.1.2 Mengecek Missing Values dan Duplikasi

Menghitung nilai kosong per kolom dan memeriksa duplikasi `video_id`. Kolom deskripsi dan bahasa audio wajar kosong pada sebagian video. `like_count` yang kosong berarti disembunyikan pemilik video, bukan nol.

In [ ]:
def count_missing(df):
    missing = df.isna() | df.astype(str).apply(lambda s: s.str.strip().eq(""))
    return pd.DataFrame({"column": df.columns, "missing": missing.sum().values,
                         "missing_pct": (100 * missing.mean().values).round(2)})


video_missing = count_missing(videos)
display(video_missing[video_missing["missing"] > 0])
print("Duplikasi video_id:", int(videos["video_id"].duplicated().sum()))

### 3.1.3 Statistik Deskriptif Views, Likes, dan Comments

In [ ]:
display(videos[["view_count", "like_count", "comment_count"]].astype("float").describe().round(1))

### 3.1.4 Distribusi Jumlah View

Histogram jumlah view dengan sumbu-x skala logaritmik, karena jumlah view biasanya sangat timpang (sedikit video viral, banyak video dengan view rendah).

In [ ]:
views = videos["view_count"].dropna().astype(float)
views = views[views > 0]
log_bins = np.logspace(0, np.log10(views.max()) + 0.1, 40).tolist()  # Bin berjarak rata pada skala log.
plt.figure(figsize=(10, 4))
plt.hist(views, bins=log_bins, color="#4C72B0", edgecolor="white")
plt.xscale("log")
plt.title("Distribusi Jumlah View Video (Skala Log)")
plt.xlabel("view_count")
plt.ylabel("Jumlah video")
save_figure("video_view_distribution")

### 3.1.5 Jumlah Video per Channel

Menampilkan jumlah video MBG per channel, termasuk channel dengan 0 video, agar terlihat bahwa channel tersebut sudah dicari tetapi tidak memiliki video yang memenuhi kriteria.

In [ ]:
video_per_channel = (df_channels[["channel_name", "category", "method"]]
                     .merge(videos.groupby("channel_name")["video_id"].count().rename("n_videos"),
                            on="channel_name", how="left")
                     .fillna({"n_videos": 0})
                     .astype({"n_videos": int})
                     .sort_values("n_videos", ascending=False))
display(video_per_channel)

plt.figure(figsize=(9, 8))
data_plot = video_per_channel.sort_values("n_videos")
plt.barh(data_plot["channel_name"], data_plot["n_videos"], color="#4C72B0")
plt.title("Jumlah Video MBG per Channel")
plt.xlabel("Jumlah video")
save_figure("video_per_channel")

### 3.1.6 Perbandingan Media Nasional dan BGN

Membandingkan jumlah video dan rata-rata engagement antara channel media nasional dan channel pemerintah (BGN), karena keduanya mewakili jenis sumber yang berbeda.

In [ ]:
category_summary = videos.groupby("category").agg(
    n_videos=("video_id", "count"),
    median_views=("view_count", "median"),
    median_comments=("comment_count", "median"),
    total_comments_api=("comment_count", "sum"),
)
display(category_summary)

### 3.1.7 Tren Jumlah Video per Bulan

In [ ]:
video_per_month = videos.set_index("published_at").resample("MS")["video_id"].count()
plt.figure(figsize=(11, 4))
plt.plot(video_per_month.index, video_per_month.values, marker="o")
plt.title("Jumlah Video MBG per Bulan (tanggal terbit)")
plt.ylabel("Jumlah video")
save_figure("video_per_month")

### 3.1.8 Lokasi Keyword MBG dan Video Siaran Langsung

Memeriksa berapa video yang memuat keyword MBG di judul dibanding yang hanya di deskripsi, serta berapa yang berasal dari siaran langsung. Video yang hanya cocok di deskripsi, terutama siaran langsung, berpotensi membahas MBG hanya sekilas sehingga perlu diperhatikan saat analisis.

In [ ]:
display(pd.crosstab(videos["mbg_match_field"], videos["was_live"], margins=True,
                    rownames=["mbg_match_field"], colnames=["was_live"]))
print("Metode pencarian:", videos["search_method"].value_counts().to_dict())

### 3.1.9 Frekuensi Kata pada Judul Video

Menghitung 20 kata yang paling sering muncul pada judul video. Stopword umum dan kata keyword MBG itu sendiri (`mbg`, `makan`, `bergizi`, `gratis`) dikecualikan agar kata lain terlihat.

In [ ]:
title_words = top_words(videos["title_clean"])
display(title_words)

### 3.1.10 Wordcloud Judul Video

In [ ]:
try:
    from wordcloud import WordCloud
    title_freq = Counter(w for t in videos["title_clean"] for w in re.findall(r"[a-z]{3,}", t.lower())
                         if w not in EDA_STOPWORDS)
    if title_freq:
        wordcloud = WordCloud(width=1000, height=500, background_color="white").generate_from_frequencies(title_freq)
        plt.figure(figsize=(12, 6))
        plt.imshow(wordcloud)
        plt.axis("off")
        plt.title("Wordcloud Judul Video")
        save_figure("wordcloud_video_titles")
except ImportError:
    print("Library wordcloud belum terpasang: pip install wordcloud")

### 3.1.11 Korelasi Views, Likes, dan Comments

Menghitung korelasi Spearman (berbasis peringkat), karena lebih tahan terhadap outlier dibanding korelasi Pearson pada data yang sangat timpang.

In [ ]:
display(videos[["view_count", "like_count", "comment_count"]].astype("float").corr(method="spearman").round(2))

### 3.1.12 Top 10 Video dengan View Tertinggi

In [ ]:
display(videos.nlargest(10, "view_count")[["channel_name", "published_at", "title", "view_count",
                                           "like_count", "comment_count"]])

## 3.2. Indikasi Isu dan Lokasi

### 3.2.1 Kemunculan Keyword Keamanan Pangan (Indikasi Awal)

Menghitung berapa video yang menyebut kelompok keyword terkait masalah keamanan dan kebersihan makanan. Ini hanya **pencocokan kata sederhana** untuk melihat apakah dataset memuat pembahasan yang relevan.

In [ ]:
FOOD_SAFETY_KEYWORDS = {
    "keracunan": r"\b(?:keracunan|diare|mual|muntah)\b",
    "makanan_basi": r"\b(?:basi|busuk|berbau|bau)\b",
    "benda_organisme_asing": r"\b(?:belatung|ulat|cacing|kecoa|lalat|rambut|plastik)\b",
    "kontaminasi_bakteri": r"\b(?:bakteri|kontaminasi|terkontaminasi|e\.?\s?coli|salmonella)\b",
    "kebersihan_higiene": r"\b(?:higiene|higienis|kebersihan|jorok|kotor|sanitasi)\b",
}

video_text = videos["title_clean"] + " " + videos["description_clean"]
keyword_summary = pd.DataFrame([
    {"keyword_group": group, "n_videos": int(video_text.str.contains(pattern, case=False, regex=True).sum())}
    for group, pattern in FOOD_SAFETY_KEYWORDS.items()
])
display(keyword_summary)

plt.figure(figsize=(9, 4))
plt.barh(keyword_summary["keyword_group"], keyword_summary["n_videos"], color="#55A868")
plt.title("Jumlah Video yang Menyebut Keyword Keamanan Pangan")
plt.xlabel("Jumlah video")
save_figure("food_safety_keywords")

### 3.2.2 Sebaran Lokasi yang Disebut

Menghitung jumlah video yang menyebut setiap provinsi (hasil bagian 2.3), termasuk video yang juga menyebut keyword keracunan. Satu video dapat menyebut lebih dari satu provinsi. Tabel disimpan ke `data/source/location_per_province.csv`. Jakarta sering muncul sebagai lokasi pernyataan pejabat, bukan lokasi kejadian.

In [ ]:
def count_by_province(df):
    provinces = df["province"].str.split("; ").explode()
    return provinces[provinces != ""].value_counts()


poisoning_pattern = r"\b(?:keracunan|diare|mual|muntah)\b"
poisoning_videos = videos[video_text.str.contains(poisoning_pattern, case=False, regex=True)]
location_per_province = (pd.DataFrame({"n_videos": count_by_province(videos),
                                       "n_videos_keracunan": count_by_province(poisoning_videos)})
                         .fillna(0).astype(int).sort_values("n_videos", ascending=False))
location_per_province.index.name = "province"
display(location_per_province)
save_csv(location_per_province.reset_index(), OUTPUT_DIR / "location_per_province.csv")

top_provinces = location_per_province.head(15).iloc[::-1]
plt.figure(figsize=(9, 6))
plt.barh(top_provinces.index, top_provinces["n_videos"], color="#4C72B0", label="Semua video")
plt.barh(top_provinces.index, top_provinces["n_videos_keracunan"], color="#C44E52", label="Video menyebut keracunan")
plt.title("15 Provinsi yang Paling Sering Disebut pada Video")
plt.xlabel("Jumlah video")
plt.legend()
save_figure("location_per_province")

### 3.2.3 Ringkasan Dataset

In [ ]:
summary = {
    "n_channels": len(df_channels),
    "n_videos": len(videos),
    "n_videos_with_location": int((videos["location_names"] != "").sum()),
    "first_video_utc": videos["published_at"].min(),
    "last_video_utc": videos["published_at"].max(),
    "collection_complete": COLLECTION_COMPLETE,
}
df_summary = pd.DataFrame(list(summary.items()), columns=["metric", "value"])
display(df_summary)
save_csv(df_summary, OUTPUT_DIR / "dataset_summary.csv")
save_csv(video_per_channel, OUTPUT_DIR / "video_per_channel.csv")

# 4. Keterbatasan dan Etika

**Keterbatasan cakupan**

* Search API bergantung pada indeks pencarian YouTube dan tidak menjamin seluruh video MBG ditemukan. Video yang membahas MBG hanya di audio tanpa menyebutnya di judul/deskripsi juga tidak terjangkau.
* Video yang dihapus atau dijadikan privat tidak dapat diambil.
* Views, likes, dan jumlah komentar adalah *snapshot* saat data diambil, bukan angka harian.
* Kata `MBG` dapat bermakna lain. Video yang hanya cocok di deskripsi (kolom `mbg_match_field`) perlu diperiksa saat analisis.
* Lokasi diekstraksi dengan pencocokan kamus wilayah, sehingga hanya menangkap nama wilayah yang **disebut** di teks dan belum tentu lokasi kejadian. Nama di luar kamus (desa, kecamatan, nama sekolah) dan salah ketik di luar daftar alias tidak terdeteksi, sedangkan nama ambigu tanpa kata konteks dilewati. Jakarta sering muncul sebagai lokasi pernyataan pejabat.

**Etika dan privasi**

* **Hanya data publik dari institusi.** Yang diambil adalah metadata video (judul, deskripsi, statistik) dari channel media nasional dan lembaga pemerintah, bukan data pribadi pengguna. Komentar dan identitas penonton tidak dikumpulkan.
* **Lewat API resmi.** Data diambil melalui YouTube Data API v3 sesuai Ketentuan Layanan YouTube API Services, dengan batas kuota per sesi agar tidak membebani layanan.
* **Rahasia tidak masuk repo.** API key dibaca dari environment variable atau file `.env` yang dikecualikan oleh `.gitignore`, dan tidak pernah dicetak.
* **Data mentah tidak dipublikasikan.** Folder `data/raw/` dan `data/source/` dikecualikan dari repo; yang dipublikasikan hanya kode, dokumentasi, dan tabel agregat untuk dashboard.
* **Retensi data.** Kebijakan YouTube API meminta data yang disimpan diperbarui atau dihapus secara berkala; dataset dipakai terbatas untuk keperluan analisis dan portofolio non-komersial.
* **Label isu.** Kategori isu keamanan pangan yang dipakai dashboard adalah input terpisah, bukan hasil notebook ini.